In [5]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
import os
import random
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt

In [6]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

seed_everything(7)

In [7]:
BASE_PATH = os.path.expanduser('~/Downloads/rock_csvs')

file_map = {
    ('Granite', '1.83 Hz'): 'S10Granite_1-83Hz_profiles.csv',
    ('Granite', '5.10 Hz'): 'S10Granite_5-10Hz_profiles.csv',
    ('Sandstone', '1.83 Hz'): 'Holstein_Sandstone_1-83Hz_profiles.csv',
    ('Sandstone', '5.10 Hz'): 'Holstein_Sandstone_5-10Hz_profiles.csv',
    ('Limestone', '1.83 Hz'): 'Leitendorf_Limestone_1-83Hz_profiles.csv',
    ('Limestone', '5.10 Hz'): 'Leitendorf_Limestone_5-10Hz_profiles.csv',
}

dfs_183, dfs_510 = [], []

for (rock, speed), filename in file_map.items():
    file_path = os.path.join(BASE_PATH, filename)

    if os.path.exists(file_path):
        df = pd.read_csv(file_path, header=None, decimal='.')
        df_numeric = df.iloc[3::6, :1060].astype(np.float32)

        if speed == '1.83 Hz':
            dfs_183.append(df_numeric)
        else:
            dfs_510.append(df_numeric)
    else:
        print(f"[WARNING] File not found: {file_path}")

In [8]:
dfs_183[0]

,0,1,2,3,4,5,6,7,8,9,...,1050,1051,1052,1053,1054,1055,1056,1057,1058,1059
3,0.33,0.33,0.33,0.33,0.33,0.33,0.33,0.33,0.34,0.34,...,0.51,0.51,0.50,0.50,0.50,0.50,0.49,0.49,0.49,0.49
9,0.32,0.32,0.32,0.32,0.32,0.32,0.32,0.31,0.31,0.31,...,0.69,0.69,0.68,0.68,0.67,0.67,0.66,0.66,0.66,0.65
15,0.40,0.40,0.40,0.39,0.39,0.39,0.39,0.39,0.39,0.39,...,0.53,0.52,0.51,0.50,0.50,0.49,0.49,0.48,0.48,0.47
21,0.32,0.32,0.33,0.33,0.33,0.34,0.34,0.34,0.35,0.35,...,0.53,0.52,0.50,0.49,0.48,0.47,0.46,0.45,0.44,0.43
27,0.38,0.38,0.37,0.37,0.37,0.36,0.36,0.35,0.35,0.35,...,0.58,0.58,0.57,0.57,0.56,0.56,0.55,0.55,0.55,0.54
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23973,0.31,0.32,0.32,0.33,0.33,0.34,0.35,0.35,0.36,0.36,...,0.62,0.61,0.61,0.60,0.60,0.59,0.59,0.58,0.58,0.57
23979,0.30,0.31,0.31,0.31,0.32,0.32,0.32,0.33,0.33,0.33,...,0.62,0.61,0.61,0.61,0.60,0.60,0.59,0.59,0.58,0.58
23985,0.70,0.71,0.72,0.73,0.74,0.75,0.76,0.77,0.78,0.79,...,0.65,0.64,0.64,0.64,0.64,0.63,0.63,0.63,0.62,0.62
23991,0.31,0.31,0.31,0.31,0.32,0.32,0.32,0.32,0.32,0.32,...,0.60,0.60,0.59,0.59,0.59,0.58,0.58,0.58,0.57,0.57


In [9]:
dfs_510[0]

,0,1,2,3,4,5,6,7,8,9,...,1050,1051,1052,1053,1054,1055,1056,1057,1058,1059
3,0.33,0.33,0.34,0.34,0.35,0.35,0.36,0.37,0.38,0.39,...,0.83,0.82,0.81,0.80,0.79,0.78,0.77,0.76,0.75,0.74
9,0.52,0.52,0.52,0.52,0.52,0.52,0.52,0.52,0.52,0.52,...,0.90,0.90,0.90,0.90,0.90,0.90,0.90,0.90,0.90,0.90
15,0.44,0.44,0.45,0.45,0.45,0.46,0.46,0.46,0.47,0.47,...,0.78,0.77,0.77,0.76,0.76,0.75,0.75,0.75,0.74,0.74
21,0.42,0.42,0.42,0.42,0.42,0.41,0.41,0.41,0.40,0.40,...,0.57,0.56,0.56,0.55,0.54,0.54,0.53,0.52,0.52,0.51
27,0.44,0.44,0.43,0.43,0.43,0.43,0.43,0.43,0.43,0.43,...,0.71,0.70,0.70,0.69,0.69,0.68,0.68,0.68,0.67,0.67
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24903,0.62,0.62,0.62,0.63,0.63,0.64,0.65,0.65,0.66,0.66,...,0.44,0.44,0.44,0.45,0.45,0.45,0.45,0.45,0.46,0.46
24909,0.43,0.43,0.43,0.43,0.43,0.43,0.43,0.43,0.43,0.43,...,0.40,0.39,0.39,0.38,0.38,0.38,0.37,0.37,0.37,0.37
24915,0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39,...,0.43,0.43,0.42,0.42,0.41,0.41,0.41,0.40,0.40,0.40
24921,0.44,0.44,0.44,0.45,0.45,0.46,0.46,0.46,0.47,0.47,...,0.40,0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39


In [10]:
class_names

['Limestone', 'Sandstone', 'Granite']

In [11]:
dataset_183 = []
for i, d in enumerate(dfs_183):
    class_num = np.array([i] * d.shape[0], dtype=np.float32)
    d = np.c_[d, class_num]  # concatenate along 2nd axis
    dataset_183.extend(d)
dataset_183 = np.array(dataset_183)

dataset_510 = []
for i, d in enumerate(dfs_510):
    class_num = np.array([i] * d.shape[0], dtype=np.float32)
    d = np.c_[d, class_num]
    dataset_510.extend(d)
dataset_510 = np.array(dataset_510)

print('1.83 Hz dataset shape:', dataset_183.shape)
print('5.10 Hz dataset shape:', dataset_510.shape)

1.83 Hz dataset shape: (12054, 1061)
5.10 Hz dataset shape: (12273, 1061)


In [12]:
X_183 = dataset_183[:, :-1]
y_183 = dataset_183[:, -1:]

X_183.shape, X_183[0, -5:], y_183.shape, np.unique(y_183)

((12054, 1060),
 array([0.5 , 0.49, 0.49, 0.49, 0.49], dtype=float32),
 (12054, 1),
 array([0., 1., 2.], dtype=float32))

In [13]:
X_510 = dataset_510[:, :-1]
y_510 = dataset_510[:, -1:]

X_510.shape, X_510[0, -5:], y_510.shape, np.unique(y_510)

((12273, 1060),
 array([0.78, 0.77, 0.76, 0.75, 0.74], dtype=float32),
 (12273, 1),
 array([0., 1., 2.], dtype=float32))

In [14]:
X_train_183, X_test_183, y_train_183, y_test_183 = sklearn.model_selection.train_test_split(
    X_183, y_183, test_size=0.2, random_state=3, stratify=y_183
)
X_train_510, X_test_510, y_train_510, y_test_510 = sklearn.model_selection.train_test_split(
    X_510, y_510, test_size=0.2, random_state=3, stratify=y_510
)

print('1.83 Hz -> Train:', X_train_183.shape, ' Test:', X_test_183.shape)
print('5.10 Hz -> Train:', X_train_510.shape, ' Test:', X_test_510.shape)

1.83 Hz -> Train: (9643, 1060)  Test: (2411, 1060)
5.10 Hz -> Train: (9818, 1060)  Test: (2455, 1060)


In [15]:
class RockDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

    def __len__(self):
        return len(self.X)


train_dataset_183 = RockDataset(X_train_183, y_train_183)
test_dataset_183  = RockDataset(X_test_183,  y_test_183)

train_dataset_510 = RockDataset(X_train_510, y_train_510)
test_dataset_510  = RockDataset(X_test_510,  y_test_510)

In [16]:
import itertools

# Define lists for each hyperparameter
batch_sizes   = [2048]
lrs           = [0.0001, 0.00001]
epochs_list   = [50]
weight_decays = [0., 1e-4]

# Generate all combinations
hyperparameters_list = []
for batch_size, lr, epochs, weight_decay in itertools.product(batch_sizes, lrs, epochs_list, weight_decays):
    hyperparameters_list.append({
        'batch_size': batch_size,
        'lr': lr,
        'epochs': epochs,
        'weight_decay': weight_decay
    })

print(f'Generated {len(hyperparameters_list)} combinations.')

Generated 4 combinations.


In [17]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm_input = nn.LayerNorm(normalized_shape=1060)
        self.conv1 = nn.Conv1d(1, 16, kernel_size=9, stride=3)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=9, stride=3)
        self.layer_norm_1 = nn.LayerNorm(normalized_shape=32)
        self.fnn1 = nn.Linear(32, len(class_names))

        self.relu        = nn.ReLU()
        self.max_pool    = nn.MaxPool1d(kernel_size=2)
        self.global_pool = nn.AdaptiveMaxPool1d(output_size=1)
        self.flatten     = nn.Flatten()
        self.dropout     = nn.Dropout(p=0.2)

    def forward(self, x):
        x = self.layer_norm_input(x)
        x = x.unsqueeze(1)
        x = self.relu(self.conv1(x))
        x = self.max_pool(x)
        x = self.relu(self.conv2(x))
        x = self.global_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.layer_norm_1(x)
        x = self.fnn1(x)
        return x


model = Model()
sum(p.numel() for p in model.parameters())

7083

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import json
import numpy as np


def run_experiments(train_dataset, test_dataset, X_test, y_test, speed_tag):

    # Runs all hyperparameter combinations for one belt speed.
    experiment_results = []
    results_dir = f'results_1d_cnn_speed_regime_comparison/{speed_tag.replace(" ", "_")}'
    os.makedirs(results_dir, exist_ok=True)

    # Select 4 fixed profiles from test set for monitoring
    monitor_indices = np.arange(4)
    X_monitor = torch.from_numpy(X_test[monitor_indices])
    y_monitor = torch.from_numpy(y_test[monitor_indices]).squeeze().long()

    print(f'Selected Monitor Indices: {monitor_indices}')
    print(f'Monitor True Labels: {[class_names[int(i)] for i in y_monitor]}')

    for idx, config in enumerate(hyperparameters_list):
        print(f"\n{'='*20}\nStarting Experiment {idx+1}\nConfig: {config}\n{'='*20}")
        config_number = idx

        # Setup Loaders
        train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True,  num_workers=0)
        test_loader  = DataLoader(test_dataset,  batch_size=config['batch_size'], shuffle=False, num_workers=0)

        # Setup Model & Optimizer
        model     = Model()
        optimizer = torch.optim.Adam(params=model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
        criterion = nn.CrossEntropyLoss()

        # Initialize best tracking for this experiment
        best_accuracy   = 0.0
        best_model_path = os.path.join(results_dir, f'best_model_exp_{idx+1}.pth')

        # TensorBoard writer — one per experiment
        tb_log_dir = os.path.join(results_dir, f'tb_exp{idx+1}_lr{config["lr"]}_wd{config["weight_decay"]}')
        writer = SummaryWriter(log_dir=tb_log_dir)

        # Training lists
        training_loss = []
        training_acc  = []
        testing_loss  = []
        testing_acc   = []

        for epoch in range(config['epochs']):
            avg_train_loss = []
            avg_train_acc  = []
            avg_test_loss  = []
            avg_test_acc   = []

            print(f'Epoch: {epoch + 1}/{config["epochs"]}')

            # Train
            model.train()
            for i, d in tqdm(enumerate(train_loader), total=len(train_loader), leave=False):
                X_batch, y_batch = d
                y_batch = y_batch.squeeze().long()
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss    = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()

                _, predicted_labels = torch.max(outputs, 1)
                correct_predictions = (predicted_labels == y_batch).sum().item()
                total_samples       = y_batch.size(0)
                accuracy            = correct_predictions / total_samples

                avg_train_loss.append(loss.item())
                avg_train_acc.append(accuracy)

            avg_acc_train  = sum(avg_train_acc)  / len(avg_train_acc)
            avg_loss_train = sum(avg_train_loss) / len(avg_train_loss)
            training_acc.append(avg_acc_train)
            training_loss.append(avg_loss_train)
            print(f'Training: Loss={avg_loss_train:.4f}, Acc={avg_acc_train:.4f}')

            # Test
            model.eval()
            all_preds, all_true = [], []
            with torch.no_grad():
                for i, d in tqdm(enumerate(test_loader), total=len(test_loader), leave=False):
                    X_batch, y_batch = d
                    y_batch = y_batch.squeeze().long()
                    outputs = model(X_batch)
                    loss    = criterion(outputs, y_batch)

                    preds = torch.max(outputs, 1)[1]
                    all_preds.extend(preds.cpu().numpy())
                    all_true.extend(y_batch.cpu().numpy())

                    acc = (preds == y_batch).sum().item() / y_batch.size(0)
                    avg_test_loss.append(loss.item())
                    avg_test_acc.append(acc)

            avg_acc_test  = sum(avg_test_acc)  / len(avg_test_acc)
            avg_loss_test = sum(avg_test_loss) / len(avg_test_loss)
            testing_acc.append(avg_acc_test)
            testing_loss.append(avg_loss_test)
            print(f'Testing: Loss={avg_loss_test:.4f}, Acc={avg_acc_test:.4f}\n')

            # Log to TensorBoard
            writer.add_scalars('Loss',     {'train': avg_loss_train, 'test': avg_loss_test}, epoch)
            writer.add_scalars('Accuracy', {'train': avg_acc_train,  'test': avg_acc_test},  epoch)

            if avg_acc_test > best_accuracy:
                best_accuracy = avg_acc_test
                torch.save(model.state_dict(), best_model_path)
                print(f' New Best Acc: {best_accuracy:.4f} - Saved Model.')

            # Check performance on monitored profiles
            model.eval()
            with torch.no_grad():
                outputs_monitor  = model(X_monitor)
                _, pred_monitor  = torch.max(outputs_monitor, 1)

                print(f'Monitor Profiles (Epoch {epoch + 1}):')
                for j, (true_cls, pred_cls) in enumerate(zip(y_monitor, pred_monitor)):
                    status = '\u2713' if true_cls == pred_cls else '\u2717'
                    print(f'  Sample {monitor_indices[j]}: True: {class_names[true_cls.item()]:<15} | Pred: {class_names[pred_cls.item()]:<15} {status}')
            print('-' * 30)

        # --- Generate Confusion Matrix ---
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        model.load_state_dict(torch.load(best_model_path, map_location=device))
        model.to(device)
        model.eval()

        all_preds, all_true = [], []
        with torch.no_grad():
            for i, d in tqdm(enumerate(test_loader), total=len(test_loader), leave=False):
                X_batch, y_batch = d
                y_batch = y_batch.squeeze().long()
                outputs = model(X_batch.to(device))
                preds   = torch.max(outputs, 1)[1]
                all_preds.extend(preds.cpu().numpy())
                all_true.extend(y_batch.cpu().numpy())

        cm = confusion_matrix(all_true, all_preds)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names)
        plt.title(f'Confusion Matrix - {speed_tag} - Exp {idx+1}')
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        plt.tight_layout()
        cm_path = os.path.join(results_dir, f'confusion_matrix_exp_{idx+1}.png')
        plt.savefig(cm_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'[SAVED] Confusion matrix -> {cm_path}')
        plt.close()

        # Close TensorBoard writer for this experiment
        writer.close()
        print(f'TensorBoard logs saved to: {tb_log_dir}')

        # Store results
        experiment_results.append({
            'config'         : config,
            'final_train_acc': training_acc[-1],
            'final_test_acc' : testing_acc[-1],
            'final_test_loss': testing_loss[-1],
            'best_model_path': best_model_path,
            'history': {
                'train_loss': training_loss,
                'train_acc' : training_acc,
                'test_loss' : testing_loss,
                'test_acc'  : testing_acc
            }
        })

    # Display Summary
    print('\n' + '='*40)
    print(f'HYPERPARAMETER TUNING RESULTS - {speed_tag}')
    print('='*40)
    for res in experiment_results:
        print(f"Config: {res['config']}")
        print(f"  -> Final Test Acc: {res['final_test_acc']:.4f}")
        print(f"  -> Final Test Loss: {res['final_test_loss']:.4f}")
        print('-' * 20)

    return experiment_results


# Run for 1.83 Hz
print('\n' + '#'*50)
print('TRAINING - 1.83 Hz')
print('#'*50)
results_183 = run_experiments(train_dataset_183, test_dataset_183, X_test_183, y_test_183, '1.83 Hz')

# Run for 5.10 Hz
print('\n' + '#'*50)
print('TRAINING - 5.10 Hz')
print('#'*50)
results_510 = run_experiments(train_dataset_510, test_dataset_510, X_test_510, y_test_510, '5.10 Hz')


##################################################
TRAINING - 1.83 Hz
##################################################
Selected Monitor Indices: [0 1 2 3]
Monitor True Labels: ['Sandstone', 'Limestone', 'Granite', 'Granite']

Starting Experiment 1
Config: {'batch_size': 2048, 'lr': 0.0001, 'epochs': 50, 'weight_decay': 0.0}


/opt/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 32 worker processes in total. Our suggested max number of worker in current system is 8 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch: 1/50


/opt/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 32 worker processes in total. Our suggested max number of worker in current system is 8 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=92, pipe_handle=106)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/miniconda3/lib/python3.13/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/miniconda3/lib/python3.13/multiprocessing/spawn.py", line 132, in _main
 

KeyboardInterrupt: 

In [ ]:
# Analyze Hyperparameter Tuning Results - 1.83 Hz
results_df_183 = pd.DataFrame([
    {
        'batch_size'     : res['config']['batch_size'],
        'lr'             : res['config']['lr'],
        'weight_decay'   : res['config']['weight_decay'],
        'final_test_acc' : res['final_test_acc'],
        'final_test_loss': res['final_test_loss']
    }
    for res in results_183
])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Hyperparameter Impact on Test Accuracy - 1.83 Hz', fontsize=16)

# Learning Rate
lrs = sorted(results_df_183['lr'].unique())
axes[0].boxplot([results_df_183[results_df_183['lr'] == lr]['final_test_acc'] for lr in lrs],
                labels=lrs)
axes[0].set_title('Learning Rate vs Test Accuracy')
axes[0].set_xlabel('Learning Rate')
axes[0].set_ylabel('Test Accuracy')

# Weight Decay
wds = sorted(results_df_183['weight_decay'].unique())
axes[1].boxplot([results_df_183[results_df_183['weight_decay'] == wd]['final_test_acc'] for wd in wds],
                labels=wds)
axes[1].set_title('Weight Decay vs Test Accuracy')
axes[1].set_xlabel('Weight Decay')
axes[1].set_ylabel('Test Accuracy')

plt.tight_layout()
_RES = 'results_1d_cnn_speed_regime_comparison'
import os as _os; _os.makedirs(_RES, exist_ok=True)
_p = _os.path.join(_RES, 'hyperparam_impact_183Hz.png')
plt.savefig(_p, dpi=150, bbox_inches='tight')
print(f'[SAVED] hyperparam_impact_183Hz.png -> {_p}')
plt.show()

# Training History Comparison - 1.83 Hz
plt.figure(figsize=(12, 6))
for res in results_183:
    config_str = f"BS={res['config']['batch_size']}, LR={res['config']['lr']}, WD={res['config']['weight_decay']}"
    plt.plot(res['history']['test_acc'], label=config_str)
plt.title('Test Accuracy History per Configuration - 1.83 Hz')
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
_RES = 'results_1d_cnn_speed_regime_comparison'
import os as _os; _os.makedirs(_RES, exist_ok=True)
_p = _os.path.join(_RES, 'accuracy_history_183Hz.png')
plt.savefig(_p, dpi=150, bbox_inches='tight')
print(f'[SAVED] accuracy_history_183Hz.png -> {_p}')
plt.show()

In [ ]:
# Loss curves - 1.83 Hz (best experiment)
best_183 = max(results_183, key=lambda x: x['final_test_acc'])
training_loss_183 = best_183['history']['train_loss']
testing_loss_183  = best_183['history']['test_loss']

plt.figure(figsize=(7, 4))
plt.plot(np.arange(len(training_loss_183)), training_loss_183, linestyle='-', color='blue', label='Training loss')
plt.plot(np.arange(len(testing_loss_183)),  testing_loss_183,  linestyle='-', color='red',  label='Testing loss')
plt.title('Loss Curves - 1.83 Hz')
plt.legend()
plt.show()

In [ ]:
# Accuracy curves - 1.83 Hz (best experiment)
training_acc_183 = best_183['history']['train_acc']
testing_acc_183  = best_183['history']['test_acc']

plt.figure(figsize=(7, 4))
plt.plot(np.arange(len(training_acc_183)), training_acc_183, linestyle='-', color='blue', label='Training acc')
plt.plot(np.arange(len(testing_acc_183)),  testing_acc_183,  linestyle='-', color='red',  label='Testing acc')
plt.title('Accuracy Curves - 1.83 Hz')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Get predictions for the test set - 1.83 Hz
y_true = []
y_pred = []

best_183    = max(results_183, key=lambda x: x['final_test_acc'])
test_loader = DataLoader(test_dataset_183, batch_size=2048, shuffle=False, num_workers=0)

model = Model()
model.load_state_dict(torch.load(best_183['best_model_path']))
model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs   = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        y_true.extend(y_batch.squeeze().tolist())
        y_pred.extend(predicted.tolist())

cm   = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap=plt.cm.Blues, xticks_rotation='vertical')
plt.title('Confusion Matrix - 1.83 Hz')
plt.show()

In [ ]:
# Analyze Hyperparameter Tuning Results - 5.10 Hz
results_df_510 = pd.DataFrame([
    {
        'batch_size'     : res['config']['batch_size'],
        'lr'             : res['config']['lr'],
        'weight_decay'   : res['config']['weight_decay'],
        'final_test_acc' : res['final_test_acc'],
        'final_test_loss': res['final_test_loss']
    }
    for res in results_510
])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Hyperparameter Impact on Test Accuracy - 5.10 Hz', fontsize=16)

# Learning Rate
lrs = sorted(results_df_510['lr'].unique())
axes[0].boxplot([results_df_510[results_df_510['lr'] == lr]['final_test_acc'] for lr in lrs],
                labels=lrs)
axes[0].set_title('Learning Rate vs Test Accuracy')
axes[0].set_xlabel('Learning Rate')
axes[0].set_ylabel('Test Accuracy')

# Weight Decay
wds = sorted(results_df_510['weight_decay'].unique())
axes[1].boxplot([results_df_510[results_df_510['weight_decay'] == wd]['final_test_acc'] for wd in wds],
                labels=wds)
axes[1].set_title('Weight Decay vs Test Accuracy')
axes[1].set_xlabel('Weight Decay')
axes[1].set_ylabel('Test Accuracy')

plt.tight_layout()
_RES = 'results_1d_cnn_speed_regime_comparison'
import os as _os; _os.makedirs(_RES, exist_ok=True)
_p = _os.path.join(_RES, 'hyperparam_impact_510Hz.png')
plt.savefig(_p, dpi=150, bbox_inches='tight')
print(f'[SAVED] hyperparam_impact_510Hz.png -> {_p}')
plt.show()

# Training History Comparison - 5.10 Hz
plt.figure(figsize=(12, 6))
for res in results_510:
    config_str = f"BS={res['config']['batch_size']}, LR={res['config']['lr']}, WD={res['config']['weight_decay']}"
    plt.plot(res['history']['test_acc'], label=config_str)
plt.title('Test Accuracy History per Configuration - 5.10 Hz')
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
_RES = 'results_1d_cnn_speed_regime_comparison'
import os as _os; _os.makedirs(_RES, exist_ok=True)
_p = _os.path.join(_RES, 'accuracy_history_510Hz.png')
plt.savefig(_p, dpi=150, bbox_inches='tight')
print(f'[SAVED] accuracy_history_510Hz.png -> {_p}')
plt.show()

In [ ]:
# Loss curves - 5.10 Hz (best experiment)
best_510 = max(results_510, key=lambda x: x['final_test_acc'])
training_loss_510 = best_510['history']['train_loss']
testing_loss_510  = best_510['history']['test_loss']

plt.figure(figsize=(7, 4))
plt.plot(np.arange(len(training_loss_510)), training_loss_510, linestyle='-', color='blue', label='Training loss')
plt.plot(np.arange(len(testing_loss_510)),  testing_loss_510,  linestyle='-', color='red',  label='Testing loss')
plt.title('Loss Curves - 5.10 Hz')
plt.legend()
plt.show()

In [ ]:
# Accuracy curves - 5.10 Hz (best experiment)
training_acc_510 = best_510['history']['train_acc']
testing_acc_510  = best_510['history']['test_acc']

plt.figure(figsize=(7, 4))
plt.plot(np.arange(len(training_acc_510)), training_acc_510, linestyle='-', color='blue', label='Training acc')
plt.plot(np.arange(len(testing_acc_510)),  testing_acc_510,  linestyle='-', color='red',  label='Testing acc')
plt.title('Accuracy Curves - 5.10 Hz')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Get predictions for the test set - 5.10 Hz
y_true = []
y_pred = []

best_510    = max(results_510, key=lambda x: x['final_test_acc'])
test_loader = DataLoader(test_dataset_510, batch_size=2048, shuffle=False, num_workers=0)

model = Model()
model.load_state_dict(torch.load(best_510['best_model_path']))
model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs      = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        y_true.extend(y_batch.squeeze().tolist())
        y_pred.extend(predicted.tolist())

cm   = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap=plt.cm.Blues, xticks_rotation='vertical')
plt.title('Confusion Matrix - 5.10 Hz')
plt.show()

In [ ]:
# Select best config - 1.83 Hz
if 'results_183' in locals() and results_183:
    best_result = max(results_183, key=lambda x: x['final_test_acc'])
    best_config = best_result['config']
    print(f"Best Config (1.83 Hz): {best_config} with Acc: {best_result['final_test_acc']:.4f}")

    # Retrain with best config
    print('Retraining best model for saving...')
    if 'X_train_183' in locals() and 'y_train_183' in locals():
        full_train_dataset = RockDataset(X_train_183, y_train_183)
        train_loader = DataLoader(full_train_dataset, batch_size=best_config['batch_size'], shuffle=True)

        model     = Model()
        optimizer = torch.optim.Adam(params=model.parameters(), lr=best_config['lr'], weight_decay=best_config['weight_decay'])
        criterion = nn.CrossEntropyLoss()

        for epoch in range(best_config['epochs']):
            model.train()
            for X_batch, y_batch in train_loader:
                y_batch = y_batch.squeeze().long()
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss    = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
        print('Retraining complete.')

        import json
        os.makedirs('results_1d_cnn_speed_regime_comparison', exist_ok=True)
        torch.save(model.state_dict(), 'results_1d_cnn_speed_regime_comparison/rock_classifier_cnn_1.83Hz.pth')
        with open('results_1d_cnn_speed_regime_comparison/class_names_cnn_1.83Hz.json', 'w') as f:
            json.dump(class_names, f)
        print('Saved rock_classifier_cnn_1.83Hz.pth and class_names_cnn_1.83Hz.json')
    else:
        print('Training data not found. Please run the data loading and splitting cells.')
else:
    print('No experiment results found. Please run the training loop.')

In [ ]:
# Select best config - 5.10 Hz
if 'results_510' in locals() and results_510:
    best_result = max(results_510, key=lambda x: x['final_test_acc'])
    best_config = best_result['config']
    print(f"Best Config (5.10 Hz): {best_config} with Acc: {best_result['final_test_acc']:.4f}")

    # Retrain with best config
    print('Retraining best model for saving...')
    if 'X_train_510' in locals() and 'y_train_510' in locals():
        full_train_dataset = RockDataset(X_train_510, y_train_510)
        train_loader = DataLoader(full_train_dataset, batch_size=best_config['batch_size'], shuffle=True)

        model     = Model()
        optimizer = torch.optim.Adam(params=model.parameters(), lr=best_config['lr'], weight_decay=best_config['weight_decay'])
        criterion = nn.CrossEntropyLoss()

        for epoch in range(best_config['epochs']):
            model.train()
            for X_batch, y_batch in train_loader:
                y_batch = y_batch.squeeze().long()
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss    = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
        print('Retraining complete.')

        import json
        torch.save(model.state_dict(), 'results_1d_cnn_speed_regime_comparison/rock_classifier_cnn_5.10Hz.pth')
        with open('results_1d_cnn_speed_regime_comparison/class_names_cnn_5.10Hz.json', 'w') as f:
            json.dump(class_names, f)
        print('Saved rock_classifier_cnn_5.10Hz.pth and class_names_cnn_5.10Hz.json')
    else:
        print('Training data not found. Please run the data loading and splitting cells.')
else:
    print('No experiment results found. Please run the training loop.')

In [ ]:
#  Combined Speed Model 
# Stack both speeds together into one dataset

dataset_combined = np.vstack([dataset_183, dataset_510])
print('Combined dataset shape:', dataset_combined.shape)

In [ ]:
X_combined = dataset_combined[:, :-1]
y_combined = dataset_combined[:, -1:]

X_combined.shape, X_combined[0, -5:], y_combined.shape, np.unique(y_combined)

In [ ]:
X_train_combined, X_test_combined, y_train_combined, y_test_combined = sklearn.model_selection.train_test_split(
    X_combined, y_combined, test_size=0.2, random_state=3, stratify=y_combined
)

print('Combined -> Train:', X_train_combined.shape, ' Test:', X_test_combined.shape)

In [ ]:
train_dataset_combined = RockDataset(X_train_combined, y_train_combined)
test_dataset_combined  = RockDataset(X_test_combined,  y_test_combined)

In [ ]:
# Run experiments for combined speed model
print('\n' + '#'*50)
print('TRAINING - Combined (1.83 Hz + 5.10 Hz)')
print('#'*50)
results_combined = run_experiments(
    train_dataset_combined, test_dataset_combined,
    X_test_combined, y_test_combined, 'combined'
)

In [ ]:
# Analyze Hyperparameter Tuning Results - Combined
results_df_combined = pd.DataFrame([
    {
        'batch_size'     : res['config']['batch_size'],
        'lr'             : res['config']['lr'],
        'weight_decay'   : res['config']['weight_decay'],
        'final_test_acc' : res['final_test_acc'],
        'final_test_loss': res['final_test_loss']
    }
    for res in results_combined
])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Hyperparameter Impact on Test Accuracy - Combined', fontsize=16)

# Learning Rate
lrs = sorted(results_df_combined['lr'].unique())
axes[0].boxplot([results_df_combined[results_df_combined['lr'] == lr]['final_test_acc'] for lr in lrs],
                labels=lrs)
axes[0].set_title('Learning Rate vs Test Accuracy')
axes[0].set_xlabel('Learning Rate')
axes[0].set_ylabel('Test Accuracy')

# Weight Decay
wds = sorted(results_df_combined['weight_decay'].unique())
axes[1].boxplot([results_df_combined[results_df_combined['weight_decay'] == wd]['final_test_acc'] for wd in wds],
                labels=wds)
axes[1].set_title('Weight Decay vs Test Accuracy')
axes[1].set_xlabel('Weight Decay')
axes[1].set_ylabel('Test Accuracy')

plt.tight_layout()
_RES = 'results_1d_cnn_speed_regime_comparison'
import os as _os; _os.makedirs(_RES, exist_ok=True)
_p = _os.path.join(_RES, 'hyperparam_impact_combined.png')
plt.savefig(_p, dpi=150, bbox_inches='tight')
print(f'[SAVED] hyperparam_impact_combined.png -> {_p}')
plt.show()

# Training History Comparison - Combined
plt.figure(figsize=(12, 6))
for res in results_combined:
    config_str = f"BS={res['config']['batch_size']}, LR={res['config']['lr']}, WD={res['config']['weight_decay']}"
    plt.plot(res['history']['test_acc'], label=config_str)
plt.title('Test Accuracy History per Configuration - Combined')
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
_RES = 'results_1d_cnn_speed_regime_comparison'
import os as _os; _os.makedirs(_RES, exist_ok=True)
_p = _os.path.join(_RES, 'accuracy_history_combined.png')
plt.savefig(_p, dpi=150, bbox_inches='tight')
print(f'[SAVED] accuracy_history_combined.png -> {_p}')
plt.show()

In [ ]:
# Loss curves - Combined (best experiment)
best_combined = max(results_combined, key=lambda x: x['final_test_acc'])
training_loss_combined = best_combined['history']['train_loss']
testing_loss_combined  = best_combined['history']['test_loss']

plt.figure(figsize=(7, 4))
plt.plot(np.arange(len(training_loss_combined)), training_loss_combined, linestyle='-', color='blue', label='Training loss')
plt.plot(np.arange(len(testing_loss_combined)),  testing_loss_combined,  linestyle='-', color='red',  label='Testing loss')
plt.title('Loss Curves - Combined')
plt.legend()
plt.show()

In [ ]:
# Accuracy curves - Combined (best experiment)
training_acc_combined = best_combined['history']['train_acc']
testing_acc_combined  = best_combined['history']['test_acc']

plt.figure(figsize=(7, 4))
plt.plot(np.arange(len(training_acc_combined)), training_acc_combined, linestyle='-', color='blue', label='Training acc')
plt.plot(np.arange(len(testing_acc_combined)),  testing_acc_combined,  linestyle='-', color='red',  label='Testing acc')
plt.title('Accuracy Curves - Combined')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Get predictions for the test set - Combined
y_true = []
y_pred = []

test_loader = DataLoader(test_dataset_combined, batch_size=2048, shuffle=False, num_workers=0)

model = Model()
model.load_state_dict(torch.load(best_combined['best_model_path']))
model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs      = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        y_true.extend(y_batch.squeeze().tolist())
        y_pred.extend(predicted.tolist())

# Compute confusion matrix
cm   = confusion_matrix(y_true, y_pred)

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap=plt.cm.Blues, xticks_rotation='vertical')
plt.title('Confusion Matrix - Combined')
plt.show()

In [ ]:
# Select best config - Combined
if 'results_combined' in locals() and results_combined:
    best_result = max(results_combined, key=lambda x: x['final_test_acc'])
    best_config = best_result['config']
    print(f"Best Config (Combined): {best_config} with Acc: {best_result['final_test_acc']:.4f}")

    # Retrain with best config
    print('Retraining best model for saving...')
    if 'X_train_combined' in locals() and 'y_train_combined' in locals():
        full_train_dataset = RockDataset(X_train_combined, y_train_combined)
        train_loader = DataLoader(full_train_dataset, batch_size=best_config['batch_size'], shuffle=True)

        model     = Model()
        optimizer = torch.optim.Adam(params=model.parameters(), lr=best_config['lr'], weight_decay=best_config['weight_decay'])
        criterion = nn.CrossEntropyLoss()

        for epoch in range(best_config['epochs']):
            model.train()
            for X_batch, y_batch in train_loader:
                y_batch = y_batch.squeeze().long()
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss    = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
        print('Retraining complete.')

        import json
        torch.save(model.state_dict(), 'results_1d_cnn_speed_regime_comparison/rock_classifier_cnn_combined.pth')
        with open('results_1d_cnn_speed_regime_comparison/class_names_cnn_combined.json', 'w') as f:
            json.dump(class_names, f)
        print('Saved rock_classifier_cnn_combined.pth and class_names_cnn_combined.json')
    else:
        print('Training data not found. Please run the data loading and splitting cells.')
else:
    print('No experiment results found. Please run the training loop.')

In [ ]:
# Final Comparison: 1.83 Hz vs 5.10 Hz vs Combined 
best_183      = max(results_183,      key=lambda x: x['final_test_acc'])
best_510      = max(results_510,      key=lambda x: x['final_test_acc'])
best_combined = max(results_combined, key=lambda x: x['final_test_acc'])

print('='*40)
print('FINAL COMPARISON')
print('='*40)
print(f'1.83 Hz  best test accuracy: {best_183["final_test_acc"]:.4f}')
print(f'5.10 Hz  best test accuracy: {best_510["final_test_acc"]:.4f}')
print(f'Combined best test accuracy: {best_combined["final_test_acc"]:.4f}')

# Bar chart
labels = ['1.83 Hz', '5.10 Hz', 'Combined']
accs   = [best_183['final_test_acc'], best_510['final_test_acc'], best_combined['final_test_acc']]
colors = ['steelblue', 'coral', 'green']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, accs, color=colors, width=0.4)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Best Test Accuracy')
ax.set_title('1D CNN — Best Accuracy per Training Strategy')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f'{acc:.4f}', ha='center', fontsize=11)
plt.tight_layout()
_RES = 'results_1d_cnn_speed_regime_comparison'
import os as _os; _os.makedirs(_RES, exist_ok=True)
_p = _os.path.join(_RES, 'final_comparison_bar.png')
plt.savefig(_p, dpi=150, bbox_inches='tight')
print(f'[SAVED] final_comparison_bar.png -> {_p}')
plt.show()

In [ ]:
# Launch TensorBoard to visualize training live
print('To view TensorBoard, run this command in your terminal:')
print()
print('  cd ~/results_1d_cnn_speed_regime_comparison && tensorboard --logdir=.')
print()
print('Then open: http://localhost:6006')
print()